# Entrenamiento preliminar YOLO11n — Laboratorio de Biotecnología UTEQ

Este notebook entrena el modelo `yolo11n.pt` con el dataset preliminar (177 imágenes, 7 clases, split 70/15/15) usando **exactamente el mismo `scripts/train.py`** del proyecto — sin cambiar la lógica, solo el lugar donde corre (esto evita el bloqueo de Smart App Control de Windows en el equipo local).

**Antes de correr nada:** en el menú de arriba, `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → selecciona **GPU (T4)** → Guardar.

Este es un entrenamiento **preliminar** (todavía no hay 80–150 fotos por clase). No es el modelo final de Android.

## 1. Verificar que hay GPU asignada

In [ ]:
!nvidia-smi

## 2. Subir el paquete `colab_training_package.zip`

El archivo pesa ~513 MB. **Se recomienda Google Drive** (más confiable que la subida directa del navegador para un archivo de este tamaño):

1. Sube manualmente `ml/colab_training_package.zip` (generado en tu equipo con `python scripts/make_colab_package.py`) a tu Google Drive, en cualquier carpeta (por ejemplo, la raíz `Mi unidad/`).
2. Corre la celda de abajo ("Opción A") y ajusta `DRIVE_ZIP_PATH` a la ruta real dentro de Drive.

Si prefieres subirlo directo desde tu computadora sin usar Drive, usa la "Opción B" en su lugar (puede ser más lento o fallar con conexiones inestables).

In [ ]:
# --- Opcion A (recomendada): copiar el zip desde Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP_PATH = '/content/drive/MyDrive/colab_training_package.zip'  # <-- ajusta la ruta si subiste a otra carpeta

!cp "$DRIVE_ZIP_PATH" /content/colab_training_package.zip
!ls -lh /content/colab_training_package.zip

In [ ]:
# --- Opcion B (alternativa): subir directo desde tu computadora ---
# Descomenta y corre esta celda EN VEZ de la de arriba si no quieres usar Drive.

# from google.colab import files
# uploaded = files.upload()  # selecciona ml/colab_training_package.zip en el dialogo
# import shutil, os
# fname = list(uploaded.keys())[0]
# shutil.move(fname, '/content/colab_training_package.zip')

## 3. Descomprimir

In [ ]:
!rm -rf /content/ml_training
!mkdir -p /content/ml_training
!unzip -q /content/colab_training_package.zip -d /content/ml_training
%cd /content/ml_training
!echo '--- estructura ---' && find . -maxdepth 2 | sort

## 4. Instalar Ultralytics (y dependencias)

In [ ]:
!pip install -q -r requirements.txt

## 5. Verificar versiones y CUDA

In [ ]:
import sys, torch, torchvision, ultralytics
print('Python:', sys.version)
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 6. Verificar que data.yaml resuelve las 7 clases y las imágenes

In [ ]:
import yaml
with open('data.yaml') as f:
    d = yaml.safe_load(f)
print('names:', d['names'])
print('nc:', len(d['names']))
!echo 'train:' && ls dataset/images/train | wc -l
!echo 'val:' && ls dataset/images/val | wc -l
!echo 'test:' && ls dataset/images/test | wc -l

## 7. Entrenar

Mismo `train.py` del proyecto, sin cambios de lógica. `--batch -1` activa AutoBatch de Ultralytics (elige el batch más grande que cabe en la memoria de la GPU) — tiene sentido acá porque SÍ hay GPU real (en el equipo local, sin GPU, el script usa batch=16 fijo en su lugar).

In [ ]:
!python scripts/train.py \
  --model yolo11n.pt \
  --epochs 100 \
  --imgsz 640 \
  --batch -1 \
  --seed 42 \
  --workers 2 \
  --project runs \
  --name yolo11n_preliminar

## 8. Localizar best.pt / last.pt y resultados

In [ ]:
!find runs/yolo11n_preliminar -maxdepth 2 | sort
!echo '--- best.pt ---' && ls -lh runs/yolo11n_preliminar/weights/best.pt
!echo '--- last.pt ---' && ls -lh runs/yolo11n_preliminar/weights/last.pt
!echo '--- results.csv (ultimas filas) ---' && tail -5 runs/yolo11n_preliminar/results.csv

## 9. Descargar `best.pt`, `last.pt` y toda la carpeta de resultados

Guarda `best.pt` en `ml/models/best.pt` en tu equipo (`ml/scripts/export_tflite.py` y `ml/scripts/copy_model_to_android.py` esperan ese archivo ahí — pero **todavía no los corras**: primero hay que revisar las métricas).

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('/content/yolo11n_preliminar_results', 'zip', 'runs/yolo11n_preliminar')

files.download('runs/yolo11n_preliminar/weights/best.pt')
files.download('runs/yolo11n_preliminar/weights/last.pt')
files.download('/content/yolo11n_preliminar_results.zip')

---
### Siguiente paso (fuera de este notebook, en tu equipo)

1. Coloca el `best.pt` descargado en `ml/models/best.pt` (y opcionalmente `last.pt` también en `ml/models/`).
2. Descomprime `yolo11n_preliminar_results.zip` dentro de `ml/runs/yolo11n_preliminar/` para tener las curvas, la matriz de confusión y `results.csv` disponibles localmente.
3. Revisa las métricas con tu asistente **antes** de correr `evaluate.py`, `export_tflite.py` o `copy_model_to_android.py` — ninguno de esos pasos está autorizado todavía.